# Pratica 07 — Test di ipotesi I

**Corso:** Statistica Applicata con Python · **Lezione 07** · **Tempo stimato:** 60-90 minuti
**Notebook di lezione:** `notebooks_clase/clase07_tests_hipotesis_i_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una cella con `# TODO`: completala.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Gli esercizi con simulazione usano tolleranze basate sull'errore standard, quindi un risultato corretto non dovrebbe fallire per caso.
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Tutti i dati sono sintetici: *esempi illustrativi*.

---
## Cella 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

## Esercizio 1 — Il t a un campione, a mano

L'obiettivo di un servizio è una media di 3,0 s per query. Si misurano 10 query (*esempio illustrativo*). H0: μ = 3,0; H1: μ ≠ 3,0. Calcola:

- `media`, `s` (deviazione standard campionaria con `ddof=1`),
- `t_obs = (media − mu0) / (s / √n)` e i suoi gradi di libertà `gl`,
- `p_valor` a due code con `stats.t.sf`,
- `rechaza_h0`: `True` se rifiuti H0 con α = 0,05.

In [ ]:
tiempos = np.array([2.7, 3.1, 2.5, 3.0, 2.8, 2.6, 3.2, 2.9, 2.4, 2.8])
mu0 = 3.0
n = len(tiempos)

media = None        # TODO
s = None            # TODO
t_obs = None        # TODO
gl = None           # TODO
p_valor = None      # TODO: due code
rechaza_h0 = None   # TODO

# Controllo automatico
assert all(v is not None for v in (media, s, t_obs, gl, p_valor, rechaza_h0)), "Completa le righe contrassegnate con TODO"
ref = stats.ttest_1samp(tiempos, mu0)
assert np.isclose(t_obs, ref.statistic), "t = (media − mu0) / (s / radice di n), con s calcolata con ddof=1"
assert gl == n - 1, "i gradi di libertà sono n − 1"
assert np.isclose(p_valor, ref.pvalue), "p = 2 * stats.t.sf(abs(t), gl)"
assert rechaza_h0 == (ref.pvalue < 0.05), "confronta il p-value con α = 0,05"
print(f"t = {t_obs:.3f}   gl = {gl}   p = {p_valor:.4f}   rifiuto H0? {rechaza_h0}")
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Due gruppi indipendenti: Welch, effetto e IC

Confronti il tempo di risposta di due server con dimensioni e variabilità diverse (*esempio illustrativo*). Calcola:

- `res`: il risultato di `stats.ttest_ind` con **Welch** (`equal_var=False`),
- `dif`: media di `grupo_a` meno media di `grupo_b`,
- `d`: la d di Cohen (differenza di medie / deviazione standard combinata),
- `ic`: l'intervallo di confidenza al 95 % (`res.confidence_interval()`),
- `significativo`: `True` se p < 0,05.

In [ ]:
rng_e2 = np.random.default_rng(2026)
grupo_a = rng_e2.normal(3.0, 0.8, 30)
grupo_b = rng_e2.normal(3.5, 1.2, 35)

res = None            # TODO: test di Welch
dif = None            # TODO
d = None              # TODO: d di Cohen con deviazione standard combinata
ic = None             # TODO
significativo = None  # TODO

# Controllo automatico
assert all(v is not None for v in (res, dif, d, ic, significativo)), "Completa le righe contrassegnate con TODO"
assert np.isclose(res.statistic, -3.0697, atol=1e-3), "usa equal_var=False (Welch)"
assert np.isclose(dif, grupo_a.mean() - grupo_b.mean()), "dif = media di A − media di B"
assert np.isclose(d, -0.7266, atol=1e-3), "d = dif / deviazione standard combinata, sp = radice(((na−1)·va + (nb−1)·vb) / (na+nb−2))"
assert ic.low < dif < ic.high, "l'IC deve contenere la differenza osservata"
assert significativo is True or significativo == True, "confronta il p-value con 0,05"
print(f"dif = {dif:.3f} s   IC 95 % = ({ic.low:.3f}, {ic.high:.3f})   d = {d:.2f}   p = {res.pvalue:.4f}")
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Appaiato o indipendente?

Si misurano 20 pagine prima e dopo un'ottimizzazione (*esempio illustrativo*): ogni pagina ha due misurazioni. Calcola:

- `p_pareado`: p-value del test appaiato (`ttest_rel`),
- `p_indep`: p-value del test di Welch **trattando i dati come indipendenti** (solo per vedere cosa succede),
- `usar_pareado`: `True` se il test appaiato è quello adatto a questi dati, `False` altrimenti.

Dopo gli assert, rifletti: perché i due p-value sono così diversi?

In [ ]:
rng_e3 = np.random.default_rng(77)
antes = rng_e3.lognormal(1.0, 0.5, 20)
despues = antes - 0.25 + rng_e3.normal(0, 0.25, 20)

p_pareado = None      # TODO
p_indep = None        # TODO
usar_pareado = None   # TODO

# Controllo automatico
assert all(v is not None for v in (p_pareado, p_indep, usar_pareado)), "Completa le righe contrassegnate con TODO"
assert np.isclose(p_pareado, 3.8e-5, rtol=0.05), "usa stats.ttest_rel(antes, despues)"
assert np.isclose(p_indep, 0.656, atol=0.005), "usa stats.ttest_ind(..., equal_var=False)"
assert usar_pareado == True, "ogni pagina ha la sua coppia prima/dopo: i dati sono appaiati"
print(f"p appaiato = {p_pareado:.2e}   p indipendente = {p_indep:.3f}")
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Test per due proporzioni

Un test di due varianti di un modulo (*esempio illustrativo*): la variante nuova converte 90 visite su 1 200 e la vecchia 66 su 1 200. Calcola:

- `z`, `p`: statistica e p-value di `proportions_ztest` (metti per prima la variante nuova),
- `dif`: differenza tra le proporzioni (nuova − vecchia),
- `ic`: tupla `(inferiore, superiore)` con l'IC al 95 % della differenza (errore standard **non combinato**, con 1,96),
- `rechaza_h0`: `True` se p < 0,05.

In [ ]:
conv = np.array([90, 66])         # nuova, vecchia
vis = np.array([1200, 1200])

z = None            # TODO
p = None            # TODO
dif = None          # TODO
ic = None           # TODO: (inferiore, superiore)
rechaza_h0 = None   # TODO

# Controllo automatico
assert all(v is not None for v in (z, p, dif, ic, rechaza_h0)), "Completa le righe contrassegnate con TODO"
assert np.isclose(z, 1.9872, atol=1e-3), "usa proportions_ztest(conv, vis) con la variante nuova per prima"
assert np.isclose(p, 0.0469, atol=1e-3), "il p-value è quello a due code restituito da proportions_ztest"
assert np.isclose(dif, 0.02), "dif = 90/1200 − 66/1200"
assert ic[0] < dif < ic[1], "l'IC deve contenere la differenza"
assert np.isclose(ic[0], 0.0003, atol=5e-4) and np.isclose(ic[1], 0.0397, atol=5e-4), "IC = dif ± 1,96 · radice(p1(1−p1)/n1 + p2(1−p2)/n2)"
assert rechaza_h0 == True, "p < 0,05"
print(f"z = {z:.3f}   p = {p:.4f}   dif = {dif:.4f}   IC 95 % = ({ic[0]:.4f}, {ic[1]:.4f})")
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Simulare il tasso di falsi positivi e la potenza

Con la simulazione (seme fisso `rng_ej`), stima:

- `tasa_fp`: la frazione di test di Welch con p < **0,01** quando **H0 è vera** (due gruppi da 25, entrambi N(0, 1)), con 5 000 ripetizioni. Dovrebbe essere vicina a 0,01.
- `potencia`: la frazione di test con p < 0,05 quando c'è un effetto reale di 0,8 (gruppo A ~ N(0, 1), gruppo B ~ N(0,8, 1)), 25 per gruppo e 5 000 ripetizioni.

Suggerimento: `stats.ttest_ind(x, y, axis=1, equal_var=False)` calcola un test per ogni riga.

In [ ]:
rng_ej = np.random.default_rng(11)
N = 5_000

tasa_fp = None    # TODO: H0 vera, α = 0,01
potencia = None   # TODO: effetto reale 0,8, α = 0,05

# Controllo automatico
assert tasa_fp is not None and potencia is not None, "Completa le righe contrassegnate con TODO"
ee_fp = np.sqrt(0.01 * 0.99 / N)
assert abs(tasa_fp - 0.01) < 4 * ee_fp, "con H0 vera, il tasso di p < α deve essere ≈ α (tolleranza: 4 errori standard)"
assert abs(potencia - 0.79) < 0.04, "con effetto 0,8 e n = 25 per gruppo la potenza teorica è ≈ 0,79"
print(f"tasso di falsi positivi (α = 0,01) = {tasa_fp:.4f}   potenza = {potencia:.3f}")
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Un rapporto onesto

Un team prova una modifica su un server con sole 12 misurazioni per gruppo (*esempio illustrativo*). Completa la funzione `informe_comparacion(a, b)`, che restituisce un dizionario con le chiavi `"dif"` (media di a − media di b), `"ic_inf"` e `"ic_sup"` (IC al 95 % della differenza con Welch), `"t"`, `"p"` e `"d"` (d di Cohen).

Poi scegli in `respuesta` l'unica interpretazione corretta del risultato:

- `"a"`: non c'è nessuna differenza reale tra i due gruppi.
- `"b"`: non c'è evidenza sufficiente per affermare una differenza, ma l'IC è ampio e non esclude effetti rilevanti; con così pochi dati il test ha poca potenza.
- `"c"`: H0 è vera con il 95 % di probabilità.

In [ ]:
def informe_comparacion(a, b):
    """Restituisce dif, IC 95 % (Welch), t, p e d di Cohen."""
    # TODO: test di Welch, differenza, IC e d di Cohen
    return None

rng_r = np.random.default_rng(910)
grupo_1 = rng_r.normal(3.0, 1.0, 12)
grupo_2 = rng_r.normal(3.5, 1.0, 12)
inf = informe_comparacion(grupo_1, grupo_2)
respuesta = None    # TODO: "a", "b" o "c"

# Controllo automatico
assert inf is not None and respuesta is not None, "Completa la funzione e la risposta (TODO)"
assert set(inf) == {"dif", "ic_inf", "ic_sup", "t", "p", "d"}, "il dizionario deve avere esattamente le chiavi dif, ic_inf, ic_sup, t, p e d"
assert np.isclose(inf["dif"], grupo_1.mean() - grupo_2.mean()), "dif = media di a − media di b"
assert np.isclose(inf["t"], -1.1355, atol=1e-3) and np.isclose(inf["p"], 0.2706, atol=1e-3), "usa Welch: stats.ttest_ind(a, b, equal_var=False)"
assert np.isclose(inf["ic_inf"], -1.4931, atol=1e-3) and np.isclose(inf["ic_sup"], 0.4440, atol=1e-3), "usa res.confidence_interval(confidence_level=0.95)"
assert np.isclose(inf["d"], -0.4636, atol=1e-3), "d = dif / deviazione standard combinata"
assert respuesta == "b", "un p-value alto non dimostra che non ci sia effetto; guarda l'IC"
print(f"dif = {inf['dif']:.3f}   IC 95 % = ({inf['ic_inf']:.3f}, {inf['ic_sup']:.3f})   d = {inf['d']:.2f}   p = {inf['p']:.3f}")
print("Sfida finale corretta ✔")

---
## Conclusione della pratica

Hai calcolato un t a mano, confrontato gruppi indipendenti (Welch) e appaiati, verificato due proporzioni, controllato con la simulazione il tasso di falsi positivi e la potenza, e redatto un rapporto con effetto, IC e p-value. Prossimo passo: le lezioni 8 e 9 (ANOVA, test non parametrici, chi-quadro e confronti multipli).

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
tiempos = np.array([2.7, 3.1, 2.5, 3.0, 2.8, 2.6, 3.2, 2.9, 2.4, 2.8])
mu0 = 3.0
n = len(tiempos)

media = tiempos.mean()
s = tiempos.std(ddof=1)
t_obs = (media - mu0) / (s / np.sqrt(n))
gl = n - 1
p_valor = 2 * stats.t.sf(abs(t_obs), gl)
rechaza_h0 = bool(p_valor < 0.05)

assert all(v is not None for v in (media, s, t_obs, gl, p_valor, rechaza_h0)), "Completa le righe contrassegnate con TODO"
ref = stats.ttest_1samp(tiempos, mu0)
assert np.isclose(t_obs, ref.statistic), "t = (media − mu0) / (s / radice di n), con s calcolata con ddof=1"
assert gl == n - 1, "i gradi di libertà sono n − 1"
assert np.isclose(p_valor, ref.pvalue), "p = 2 * stats.t.sf(abs(t), gl)"
assert rechaza_h0 == (ref.pvalue < 0.05), "confronta il p-value con α = 0,05"
print(f"t = {t_obs:.3f}   gl = {gl}   p = {p_valor:.4f}   rifiuto H0? {rechaza_h0}")
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
rng_e2 = np.random.default_rng(2026)
grupo_a = rng_e2.normal(3.0, 0.8, 30)
grupo_b = rng_e2.normal(3.5, 1.2, 35)

res = stats.ttest_ind(grupo_a, grupo_b, equal_var=False)
dif = grupo_a.mean() - grupo_b.mean()
na, nb = len(grupo_a), len(grupo_b)
sp = np.sqrt(((na - 1) * grupo_a.var(ddof=1) + (nb - 1) * grupo_b.var(ddof=1)) / (na + nb - 2))
d = dif / sp
ic = res.confidence_interval(confidence_level=0.95)
significativo = bool(res.pvalue < 0.05)

assert all(v is not None for v in (res, dif, d, ic, significativo)), "Completa le righe contrassegnate con TODO"
assert np.isclose(res.statistic, -3.0697, atol=1e-3), "usa equal_var=False (Welch)"
assert np.isclose(dif, grupo_a.mean() - grupo_b.mean()), "dif = media di A − media di B"
assert np.isclose(d, -0.7266, atol=1e-3), "d = dif / deviazione standard combinata, sp = radice(((na−1)·va + (nb−1)·vb) / (na+nb−2))"
assert ic.low < dif < ic.high, "l'IC deve contenere la differenza osservata"
assert significativo is True or significativo == True, "confronta il p-value con 0,05"
print(f"dif = {dif:.3f} s   IC 95 % = ({ic.low:.3f}, {ic.high:.3f})   d = {d:.2f}   p = {res.pvalue:.4f}")
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
rng_e3 = np.random.default_rng(77)
antes = rng_e3.lognormal(1.0, 0.5, 20)
despues = antes - 0.25 + rng_e3.normal(0, 0.25, 20)

p_pareado = stats.ttest_rel(antes, despues).pvalue
p_indep = stats.ttest_ind(antes, despues, equal_var=False).pvalue
usar_pareado = True

assert all(v is not None for v in (p_pareado, p_indep, usar_pareado)), "Completa le righe contrassegnate con TODO"
assert np.isclose(p_pareado, 3.8e-5, rtol=0.05), "usa stats.ttest_rel(antes, despues)"
assert np.isclose(p_indep, 0.656, atol=0.005), "usa stats.ttest_ind(..., equal_var=False)"
assert usar_pareado == True, "ogni pagina ha la sua coppia prima/dopo: i dati sono appaiati"
print(f"p appaiato = {p_pareado:.2e}   p indipendente = {p_indep:.3f}")
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
conv = np.array([90, 66])
vis = np.array([1200, 1200])

z, p = proportions_ztest(conv, vis)
p1, p2 = conv / vis
dif = p1 - p2
ee = np.sqrt(p1 * (1 - p1) / vis[0] + p2 * (1 - p2) / vis[1])
ic = (dif - 1.96 * ee, dif + 1.96 * ee)
rechaza_h0 = bool(p < 0.05)

assert all(v is not None for v in (z, p, dif, ic, rechaza_h0)), "Completa le righe contrassegnate con TODO"
assert np.isclose(z, 1.9872, atol=1e-3), "usa proportions_ztest(conv, vis) con la variante nuova per prima"
assert np.isclose(p, 0.0469, atol=1e-3), "il p-value è quello a due code restituito da proportions_ztest"
assert np.isclose(dif, 0.02), "dif = 90/1200 − 66/1200"
assert ic[0] < dif < ic[1], "l'IC deve contenere la differenza"
assert np.isclose(ic[0], 0.0003, atol=5e-4) and np.isclose(ic[1], 0.0397, atol=5e-4), "IC = dif ± 1,96 · radice(p1(1−p1)/n1 + p2(1−p2)/n2)"
assert rechaza_h0 == True, "p < 0,05"
print(f"z = {z:.3f}   p = {p:.4f}   dif = {dif:.4f}   IC 95 % = ({ic[0]:.4f}, {ic[1]:.4f})")
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
rng_ej = np.random.default_rng(11)
N = 5_000

x = rng_ej.normal(0, 1, (N, 25))
y = rng_ej.normal(0, 1, (N, 25))
tasa_fp = (stats.ttest_ind(x, y, axis=1, equal_var=False).pvalue < 0.01).mean()

x2 = rng_ej.normal(0, 1, (N, 25))
y2 = rng_ej.normal(0.8, 1, (N, 25))
potencia = (stats.ttest_ind(x2, y2, axis=1, equal_var=False).pvalue < 0.05).mean()

assert tasa_fp is not None and potencia is not None, "Completa le righe contrassegnate con TODO"
ee_fp = np.sqrt(0.01 * 0.99 / N)
assert abs(tasa_fp - 0.01) < 4 * ee_fp, "con H0 vera, il tasso di p < α deve essere ≈ α (tolleranza: 4 errori standard)"
assert abs(potencia - 0.79) < 0.04, "con effetto 0,8 e n = 25 per gruppo la potenza teorica è ≈ 0,79"
print(f"tasso di falsi positivi (α = 0,01) = {tasa_fp:.4f}   potenza = {potencia:.3f}")
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def informe_comparacion(a, b):
    """Restituisce dif, IC 95 % (Welch), t, p e d di Cohen."""
    res = stats.ttest_ind(a, b, equal_var=False)
    ic = res.confidence_interval(confidence_level=0.95)
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    dif = np.mean(a) - np.mean(b)
    return {"dif": dif, "ic_inf": ic.low, "ic_sup": ic.high, "t": res.statistic, "p": res.pvalue, "d": dif / sp}

rng_r = np.random.default_rng(910)
grupo_1 = rng_r.normal(3.0, 1.0, 12)
grupo_2 = rng_r.normal(3.5, 1.0, 12)
inf = informe_comparacion(grupo_1, grupo_2)
respuesta = "b"

assert inf is not None and respuesta is not None, "Completa la funzione e la risposta (TODO)"
assert set(inf) == {"dif", "ic_inf", "ic_sup", "t", "p", "d"}, "il dizionario deve avere esattamente le chiavi dif, ic_inf, ic_sup, t, p e d"
assert np.isclose(inf["dif"], grupo_1.mean() - grupo_2.mean()), "dif = media di a − media di b"
assert np.isclose(inf["t"], -1.1355, atol=1e-3) and np.isclose(inf["p"], 0.2706, atol=1e-3), "usa Welch: stats.ttest_ind(a, b, equal_var=False)"
assert np.isclose(inf["ic_inf"], -1.4931, atol=1e-3) and np.isclose(inf["ic_sup"], 0.4440, atol=1e-3), "usa res.confidence_interval(confidence_level=0.95)"
assert np.isclose(inf["d"], -0.4636, atol=1e-3), "d = dif / deviazione standard combinata"
assert respuesta == "b", "un p-value alto non dimostra che non ci sia effetto; guarda l'IC"
print(f"dif = {inf['dif']:.3f}   IC 95 % = ({inf['ic_inf']:.3f}, {inf['ic_sup']:.3f})   d = {inf['d']:.2f}   p = {inf['p']:.3f}")
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (p-value, errore di tipo I/II, potenza, appaiato o indipendente, dimensione dell'effetto)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*